In [1]:
import threading
import time

# Shared variables
data_bersama = 0
jumlah_reader = 0
jumlah_writer_menunggu = 0

# Primitif sinkronisasi
mutex_reader = threading.Lock()
mutex_writer = threading.Lock()
antrean_baca = threading.Lock()   # Mengunci reader baru jika ada writer
akses_resource = threading.Lock() # Kunci utama untuk tulis/baca data

log_kejadian = []

def reader(id_r):
    global jumlah_reader
    time.sleep(0.05) # Simulasi reader datang setelah writer mulai antre
    
    # Reader harus melewati antrean_baca dulu. Jika ada writer menunggu,
    # antrean_baca dikunci sehingga reader baru tertahan di sini.
    with antrean_baca:
        with mutex_reader:
            jumlah_reader += 1
            if jumlah_reader == 1:
                akses_resource.acquire()

    # Proses membaca data
    log_kejadian.append(f"[Reader {id_r}] Mulai baca (data={data_bersama})")
    time.sleep(0.03)
    log_kejadian.append(f"[Reader {id_r}] Selesai baca")

    with mutex_reader:
        jumlah_reader -= 1
        if jumlah_reader == 0:
            akses_resource.release()

def writer(id_w):
    global jumlah_writer_menunggu, data_bersama
    
    with mutex_writer:
        jumlah_writer_menunggu += 1
        if jumlah_writer_menunggu == 1:
            # Writer pertama mengunci pintu masuk bagi reader baru
            antrean_baca.acquire()

    with akses_resource:
        data_bersama += 10
        log_kejadian.append(f"--> [Writer {id_w}] MENULIS data={data_bersama}")
        time.sleep(0.04)

    with mutex_writer:
        jumlah_writer_menunggu -= 1
        if jumlah_writer_menunggu == 0:
            # Buka kembali pintu bagi reader setelah semua writer selesai
            antrean_baca.release()

# Skenario uji: 1 reader awal, lalu writer datang terus-menerus, 
# disusul reader baru di tengah jalan
threads = [
    threading.Thread(target=reader, args=(1,)),
    threading.Thread(target=writer, args=(1,)),
    threading.Thread(target=writer, args=(2,)),
    threading.Thread(target=writer, args=(3,)),
    threading.Thread(target=reader, args=(2,)),
    threading.Thread(target=reader, args=(3,))
]

for t in threads:
    t.start()
    time.sleep(0.01)

for t in threads:
    t.join()

print("=== URUTAN EKSEKUSI READERS-WRITERS (PRIORITAS WRITER) ===")
for baris in log_kejadian:
    print(baris)

=== URUTAN EKSEKUSI READERS-WRITERS (PRIORITAS WRITER) ===
--> [Writer 1] MENULIS data=10
--> [Writer 2] MENULIS data=20
--> [Writer 3] MENULIS data=30
[Reader 1] Mulai baca (data=30)
[Reader 2] Mulai baca (data=30)
[Reader 3] Mulai baca (data=30)
[Reader 1] Selesai baca
[Reader 2] Selesai baca
[Reader 3] Selesai baca


In [ ]:
Pada kode di atas, variabel `antrean_baca` bertindak sebagai gerbang 
bagi reader baru. Begitu ada minimal satu writer yang masuk antrean 
(`jumlah_writer_menunggu == 1`), writer langsung mengunci `antrean_baca`.

Akibatnya, Reader 2 dan Reader 3 yang datang belakangan tidak 
diizinkan masuk meskipun Reader 1 sudah selesai membaca. Semua writer 
(Writer 1, 2, dan 3) akan didahulukan sampai habis.